# Week 1 Day 4 - Fewer tokens, same or better accuracy

Burak Gulbol's take on the Day 4 challenge.

Ed's short prompt cuts the tokens a lot, but it also drops the text of the documents. Some claims are decided by exactly that text: alcohol in the clinical notes (CLM-1010), the history section of a discharge summary (CLM-1016), and carrier letters that confirm a delay without its cause (CLM-1023, CLM-1025).

So the idea here is: cut what the model doesn't need, keep what it does, and let code do the simple facts.

| Variant | What changes from Ed's short prompt |
|---|---|
| `ed` | Ed's prompt, as in the lab |
| `lean` | Only the claimant's plan column of the Limits table, only the rows for this benefit, only the add-ons the policy has, shorter instructions, no `summary` field |
| `lean_docs` | `lean` plus the document text back |
| `lean_docs_facts` | `lean_docs` plus three facts worked out in code: days from incident to filing, where the incident falls against the trip dates, and the total of the items |

Then the best variant on more models, on OpenRouter and on my own GPU (GTX 1660 Ti, 6 GB).

In [ ]:
import json
import os
import time
from datetime import date
from pathlib import Path
from typing import Literal

import httpx
import ollama
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from openai import ContentFilterFinishReasonError, LengthFinishReasonError, OpenAI
from pydantic import BaseModel, Field, ValidationError
from tqdm.auto import tqdm

In [ ]:
load_dotenv(override=True)
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=os.getenv("OPENROUTER_API_KEY"), timeout=120, max_retries=5)

TINY = False            # True for a quick check on 4 claims
REPEATS = 2             # results move between runs, so each OpenRouter run is done twice
RUN_OPENROUTER = True   # False to skip Parts 1 and 2 when their results are already saved
RESULTS_FILE = Path("week1day4_results.csv")   # every answer is appended here as it comes in

## The data, as in the lab

In [ ]:
DATA = Path("../../data")


def load(name):
    return json.loads((DATA / name).read_text(encoding="utf-8"))


labels = {d["claim_id"]: d for d in load("evals/decisions.json")}
claims = [c for c in load("claims.json") if labels[c["claim_id"]]["difficulty"] != "manual"]
if TINY:
    claims = [c for c in claims if labels[c["claim_id"]]["tiny"]]
policies = {p["policy_id"]: p for p in load("policies.json")}

short_wording = (DATA / "policy_wording_short.md").read_text(encoding="utf-8")
parts = short_wording.split("\n## ")
opening = parts[0].strip()
sections = {part.split("\n")[0].strip(): "## " + part.strip() for part in parts[1:]}

print(f"{len(claims)} claims, sections: {list(sections)}")

## Ed's prompt (`ed`)

Copied from the lab so the comparison is fair.

In [ ]:
BENEFITS = {
    "trip_cancellation": "Cancellation and Interruption",
    "trip_interruption": "Cancellation and Interruption",
    "trip_delay": "Trip Delay",
    "baggage_delay": "Baggage Delay",
    "baggage": "Baggage",
    "passport": "Passport",
    "emergency_medical": "Emergency Medical",
}

ED_INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using the Payment section.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""


def ed_messages(claim):
    policy = policies[claim["policy_id"]]
    headings = ["Limits", BENEFITS[claim["claim_type"]]]
    if policy["add_ons"]:
        headings.append("Add-ons")
    headings += ["Never covered", "Payment", "Claims"]
    wording = "\n\n".join([opening] + [sections[h] for h in headings])
    record = {k: v for k, v in policy.items() if k not in ["policy_id", "customer_id", "sold_through", "status"]}
    details = {k: v for k, v in claim.items() if k not in ["policy_id", "customer_id", "status"]}
    details["documents"] = [{"name": d["name"], "type": d["type"]} for d in claim["documents"]]
    return [
        {"role": "system", "content": f"{ED_INSTRUCTIONS}\n{wording}\n"},
        {"role": "user", "content": f"# Policy record\n\n{json.dumps(record)}\n\n# Claim\n\n{json.dumps(details)}\n"},
    ]

## My prompt

**Limits for this plan and this benefit only.** The Limits table has three plan columns and ten rows. A claim needs one column and two to four rows. This also removes a way to go wrong: reading the wrong plan's column (CLM-1013 is a decline only because of the plan).

**Only the add-ons the policy has.** Most policies have none. A policy with Electronics Plus doesn't need the Adventure Sports paragraph.

In [ ]:
LIMIT_ROWS = {
    "Cancellation and Interruption": ["Cancellation and Interruption maximum"],
    "Trip Delay": ["Trip Delay waiting period, hours", "Trip Delay maximum"],
    "Baggage Delay": ["Baggage Delay waiting period, hours", "Baggage Delay maximum"],
    "Baggage": ["Baggage maximum", "Per-item limit", "Deductible"],
    "Passport": ["Passport maximum"],
    "Emergency Medical": ["Emergency Medical maximum", "Deductible"],
}

ADD_ONS = {
    "adventure_sports": "Adventure Sports",
    "pre_existing_waiver": "Pre-Existing Condition Waiver",
    "electronics_plus": "Electronics Plus",
}


def limits_table():
    """The Limits table as {row: {plan: amount}}."""
    lines = [line for line in sections["Limits"].splitlines() if line.startswith("|")]
    plans = [cell.strip().lower() for cell in lines[0].strip("|").split("|")[1:]]
    table = {}
    for line in lines[2:]:
        cells = [cell.strip() for cell in line.strip("|").split("|")]
        table[cells[0]] = dict(zip(plans, cells[1:]))
    return table


LIMITS = limits_table()


def plan_limits(plan, benefit):
    rows = [f"- {row}: {LIMITS[row][plan]}" for row in LIMIT_ROWS[benefit]]
    note = "\nThe deductible comes off once per claim." if "Deductible" in LIMIT_ROWS[benefit] else ""
    return f"## Limits on the {plan.title()} plan (US dollars)\n\n" + "\n".join(rows) + note


def add_on_text(policy):
    """The sentences of the Add-ons section for the add-ons this policy has."""
    sentences = sections["Add-ons"].split("\n", 1)[1].strip().split(". ")
    kept = [s.rstrip(".") + "." for s in sentences if any(s.startswith(ADD_ONS[a]) for a in policy["add_ons"])]
    return "## Add-ons on this policy\n\n" + " ".join(kept) if kept else ""


print(plan_limits("premium", "Baggage"))
print(add_on_text({"add_ons": ["electronics_plus"]}))

**Facts that code can work out.** "Crisp ifs go to code, fuzzy ifs to the model." Counting days between two dates is a crisp if, and small models get it wrong. I don't let code decide anything; it only states the facts.

Where the incident falls is said as before, during or after the trip, not as "covered or not". A cancellation happens before the trip by nature, and I don't want the code to push the model to a decline.

In [ ]:
def days(a, b):
    return (date.fromisoformat(b) - date.fromisoformat(a)).days


def facts(claim, policy):
    trip = policy["trip"]
    incident = claim["incident_date"]
    if incident < trip["departs_on"]:
        where = f"before the trip, {days(incident, trip['departs_on'])} days before departure"
    elif incident > trip["returns_on"]:
        where = f"after the trip, {days(trip['returns_on'], incident)} days after the return"
    else:
        where = "during the trip"
    total = sum(item["amount"] for item in claim["items"])
    return (
        "# Facts worked out for you\n\n"
        f"- Filed {days(incident, claim['submitted_on'])} days after the incident\n"
        f"- The incident was {where}\n"
        f"- The items add up to ${total:,.2f}, and ${claim['amount_claimed']:,.2f} is claimed"
    )

**Shorter instructions and a leaner claim.** The instructions say the same in fewer words. `summary` repeats the story, so it goes. The document text is the variable under test.

In [ ]:
INSTRUCTIONS = """You decide travel insurance claims for UnderCover, using only the wording and the records below.
- approve: covered and the required documents are there. Work out the payout with the Payment rules.
- decline: not covered.
- refer: a required document is missing or incomplete, the story and the documents disagree, or the wording does not settle it."""


def lean_messages(claim, docs=False, add_facts=False):
    policy = policies[claim["policy_id"]]
    benefit = BENEFITS[claim["claim_type"]]
    wording = [plan_limits(policy["plan"], benefit), sections[benefit], add_on_text(policy),
               sections["Never covered"], sections["Payment"], sections["Claims"]]
    record = {k: v for k, v in policy.items() if k not in ["policy_id", "customer_id", "sold_through", "status"]}
    details = {k: v for k, v in claim.items() if k not in ["claim_id", "policy_id", "customer_id", "status", "summary"]}
    details["documents"] = [
        {"name": d["name"], "type": d["type"], **({"text": d["content"]} if docs else {})} for d in claim["documents"]
    ]
    user = f"# Policy record\n\n{json.dumps(record)}\n\n# Claim\n\n{json.dumps(details)}"
    if add_facts:
        user += "\n\n" + facts(claim, policy)
    system = INSTRUCTIONS + "\n\n" + "\n\n".join(part for part in wording if part)
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


VARIANTS = {
    "ed": ed_messages,
    "lean": lean_messages,
    "lean_docs": lambda claim: lean_messages(claim, docs=True),
    "lean_docs_facts": lambda claim: lean_messages(claim, docs=True, add_facts=True),
}

for message in VARIANTS["lean_docs_facts"](claims[0]):
    print(message["content"], "\n")

A rough size check before spending anything: characters divided by four is close enough to tokens for English. The real counts come back from the provider in the runs.

In [ ]:
sizes = pd.DataFrame(
    {name: [sum(len(m["content"]) for m in build(c)) // 4 for c in claims] for name, build in VARIANTS.items()},
    index=[c["claim_id"] for c in claims],
)
sizes.describe().loc[["mean", "min", "max"]].round()

## Calling the models, as in the lab

In [ ]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")


def ask(model, effort, messages, ignore=None):
    provider = {"require_parameters": True}
    if ignore:
        provider["ignore"] = ignore
    for attempt in range(1, 4):
        try:
            return openrouter.chat.completions.parse(
                model=model, messages=messages, response_format=ClaimDecision,
                reasoning_effort=effort, extra_body={"provider": provider},
            )
        except (LengthFinishReasonError, ContentFilterFinishReasonError, ValidationError) as error:
            print(f"{type(error).__name__} on try {attempt}, so we ask again")
    raise RuntimeError(f"No answer from {model} after 3 tries")


def decide(claim, model, effort, variant, ignore=None):
    start = time.time()
    response = ask(model, effort, VARIANTS[variant](claim), ignore)
    answer = response.choices[0].message.parsed
    return {
        "model": model.split("/")[-1], "where": "OpenRouter", "variant": variant,
        "claim_id": claim["claim_id"], "decision": answer.decision, "payout": answer.payout,
        "rationale": answer.rationale, "provider": response.provider,
        "input_tokens": response.usage.prompt_tokens, "output_tokens": response.usage.completion_tokens,
        "cost": response.usage.cost, "seconds": round(time.time() - start, 1),
    }


LOCAL_MAX_TOKENS = 800   # a good answer is 150-400 tokens; this stops a rambling one


def decide_locally(claim, model, variant):
    start = time.time()
    response = ollama.chat(
        model=model, messages=VARIANTS[variant](claim), format=ClaimDecision.model_json_schema(),
        think=False, options={"num_ctx": 8192, "num_predict": LOCAL_MAX_TOKENS},
    )
    if response.done_reason == "length":
        raise ValueError(f"cut off at {response.eval_count} tokens")
    answer = ClaimDecision.model_validate_json(response.message.content)
    return {
        "model": model, "where": "This computer", "variant": variant,
        "claim_id": claim["claim_id"], "decision": answer.decision, "payout": answer.payout,
        "rationale": answer.rationale, "provider": "ollama",
        "input_tokens": response.prompt_eval_count, "output_tokens": response.eval_count,
        "done_reason": response.done_reason, "cost": 0.0, "seconds": round(time.time() - start, 1),
    }


def score(answer, run):
    label = labels[answer["claim_id"]]
    return {
        **answer, "run": run,
        "decision_right": answer["decision"] == label["decision"],
        "right": answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1,
    }


COLUMNS = ["model", "where", "variant", "claim_id", "run", "decision", "payout", "decision_right", "right",
           "input_tokens", "output_tokens", "done_reason", "cost", "seconds", "provider", "rationale", "error"]


def keep(row):
    """Append one answer to the results file straight away, so stopping a run loses nothing."""
    frame = pd.DataFrame([row]).reindex(columns=COLUMNS)
    frame.to_csv(RESULTS_FILE, mode="a", header=not RESULTS_FILE.exists(), index=False)


def answered():
    """The (model, variant, claim, run) answers already saved in this folder, so a stopped run can carry on."""
    done = set()
    for path in Path(".").glob("week1day4_results*.csv"):
        saved = pd.read_csv(path)
        done |= set(zip(saved["model"], saved["variant"], saved["claim_id"], saved["run"].astype(int)))
    return done


def experiment(model, variant, effort="low", local=False, ignore=None, repeats=None):
    """Run one model on one prompt variant, a few times, and keep every answer.

    A reply that can't be used counts as a wrong answer, not a skipped one.
    Answers already saved are not asked again, so you can stop and carry on later.
    """
    results = []
    done = answered()
    name = model.split("/")[-1]
    for run in range(1, (repeats or REPEATS) + 1):
        todo = [claim for claim in claims if (name, variant, claim["claim_id"], run) not in done]
        for claim in tqdm(todo, desc=f"{model} / {variant} / run {run}"):
            try:
                answer = decide_locally(claim, model, variant) if local else decide(claim, model, effort, variant, ignore)
                row = score(answer, run)
            except Exception as error:
                print(f"{claim['claim_id']}: {type(error).__name__}: {error}")
                row = {"model": name, "where": "This computer" if local else "OpenRouter",
                       "variant": variant, "claim_id": claim["claim_id"], "run": run, "decision": "no answer",
                       "decision_right": False, "right": False, "error": f"{type(error).__name__}: {error}"}
            keep(row)
            results.append(row)
    if local:
        ollama.generate(model=model, keep_alive=0)
    frame = pd.DataFrame(results)
    if len(frame):
        print(f"{model} / {variant}: decisions {frame.decision_right.mean():.0%}, with payout {frame.right.mean():.0%}, "
              f"{frame.input_tokens.mean():.0f} input tokens a claim, ${frame.cost.sum():.4f}")
    return frame

## Part 1: the four prompts on one model

DeepSeek V4.1 Flash at low reasoning, the lab's cheapest good model. Four prompts, 20 claims, twice: about 160 calls for a few cents.

In [ ]:
if RUN_OPENROUTER:
    for variant in VARIANTS:
        experiment("deepseek/deepseek-v4.1-flash", variant)

## Part 2: more models on OpenRouter

Instead of guessing model names, I ask OpenRouter for the open-weight models (the ones with a Hugging Face id) that support Structured Outputs and cost under $0.50 per million input tokens, and take the newest few. Change `EXTRA_MODELS` by hand if you want others.

In [ ]:
catalog = httpx.get(f"{OPENROUTER_BASE_URL}/models", timeout=30).json()["data"]
LAB_MODELS = {"xiaomi/mimo-v2.6-pro", "z-ai/glm-5.3-flash", "moonshotai/kimi-k3",
              "deepseek/deepseek-v4.1-flash", "nvidia/nemotron-3-ultra-550b-a55b", "mistralai/mistral-medium-3-5"}

candidates = pd.DataFrame([
    {
        "id": m["id"],
        "input $/M": float(m["pricing"]["prompt"]) * 1e6,
        "output $/M": float(m["pricing"]["completion"]) * 1e6,
        "context": m.get("context_length"),
        "created": pd.to_datetime(m["created"], unit="s").date(),
    }
    for m in catalog
    if m.get("hugging_face_id") and "structured_outputs" in (m.get("supported_parameters") or [])
    and ":" not in m["id"] and m["id"] not in LAB_MODELS and 0 < float(m["pricing"]["prompt"]) * 1e6 < 0.5
]).sort_values("created", ascending=False)
candidates.head(15)

In [ ]:
EXTRA_MODELS = list(candidates["id"].head(3))
EXTRA_MODELS

Each new model gets Ed's prompt and my best one, so we see whether the gain carries over to other models. I use `lean_docs_facts` here; if Part 1 says another variant is better, change `BEST`.

Not every model has a reasoning setting, so these run with it unset.

In [ ]:
BEST = "lean_docs_facts"

if RUN_OPENROUTER:
    for model in EXTRA_MODELS:
        for variant in ["ed", BEST]:
            experiment(model, variant, effort=None)

## Part 3: on my own GPU with Ollama

`qwen3.5:4b` from the lab, plus two sizes around it:

- `qwen3.5:2b`: smaller and faster, to see how far down we can go
- `qwen3.5:9b`: 6.6 GB, which doesn't fit in my 6 GB of video memory. I left it commented out and ran a 9B on a rented GPU instead (see the write-up)

My prompts carry the document text, so they are longer. I check that the longest one still fits in `num_ctx` (8,192), because Ollama cuts a longer prompt without an error.

**What I learned running this:** my first local run slowed from 90 to 140 seconds a claim. `ollama ps` said 100% GPU and the answers were a normal length, so I logged the speed of every answer: 25 tokens a second at the start, 3 tokens a second twenty minutes later. `nvidia-smi` showed why: 91°C and `SW Thermal Slowdown: Active`, with the laptop GPU at 28 W of its 60 W. The card throttles under a long run, which a one-off speed test never shows.

So this part runs once rather than twice, caps an answer at 800 tokens (`num_predict`, and a cut-off answer counts as wrong), and skips answers already saved, so I can stop, let the card cool down and carry on.

In [ ]:
longest = max(claims, key=lambda c: len(VARIANTS[BEST](c)[0]["content"]) + len(VARIANTS[BEST](c)[1]["content"]))
check = ollama.chat(model="qwen3.5:4b", messages=VARIANTS[BEST](longest), think=False, options={"num_ctx": 8192, "num_predict": 1})
print(f"Longest prompt: {longest['claim_id']}, {check.prompt_eval_count:,} tokens of 8,192")

In [ ]:
LOCAL_MODELS = ["qwen3.5:4b", "qwen3.5:2b"]
for model in LOCAL_MODELS:
    print(ollama.pull(model).status)

In [ ]:
for model in LOCAL_MODELS:
    for variant in ["ed", BEST]:
        experiment(model, variant, local=True, repeats=1)

In [ ]:
# Optional and slow: 6.6 GB doesn't fit in 6 GB of video memory, so part of it runs on the CPU
# print(ollama.pull("qwen3.5:9b").status)
# experiment("qwen3.5:9b", BEST, local=True, repeats=1)

## Part 4: results

In [ ]:
# Every results file in this folder, so a backup from an earlier run counts too
results = pd.concat([pd.read_csv(path) for path in sorted(Path(".").glob("week1day4_results*.csv"))], ignore_index=True)
results = results.drop_duplicates(subset=["model", "variant", "claim_id", "run", "decision", "payout", "rationale"])
summary = (
    results.groupby(["model", "where", "variant"])
    .agg(
        decisions_right=("decision_right", "mean"),
        with_payout=("right", "mean"),
        input_tokens=("input_tokens", "mean"),
        cost_a_claim=("cost", "mean"),
        seconds_a_claim=("seconds", "mean"),
        output_tokens=("output_tokens", "mean"),
        answers=("claim_id", "count"),
    )
    .reset_index()
)
for column in ["decisions_right", "with_payout"]:
    summary[column] = (100 * summary[column]).round()
summary["input_tokens"] = summary["input_tokens"].round()
summary["cost_a_claim"] = summary["cost_a_claim"].round(5)
summary["seconds_a_claim"] = summary["seconds_a_claim"].round(1)
summary.sort_values(["model", "with_payout"], ascending=[True, False])

### Accuracy against input tokens

Top left is best: right more often, with fewer tokens. Remember the baseline: approving everything gets 60% of the decisions (not the payouts) right.

In [ ]:
fig = px.scatter(
    summary, x="input_tokens", y="with_payout", color="variant", symbol="where",
    hover_data=["model", "decisions_right", "cost_a_claim"], text="model",
    title="Decision and payout right (%) against input tokens a claim", height=600,
)
fig.update_traces(marker_size=13, textposition="top center")
fig.update_yaxes(range=[0, 100])
fig.show()

### Which claims move

A score hides which claims changed. This shows, for DeepSeek, how often each claim was right (decision and payout) under each prompt, across the runs.

In [ ]:
deepseek = results[results.model == "deepseek-v4.1-flash"]
by_claim = deepseek.pivot_table(index="claim_id", columns="variant", values="right", aggfunc="mean")
by_claim["handler"] = [f"{labels[c]['decision']} {labels[c]['payout']:,.0f}" for c in by_claim.index]
by_claim["tests"] = [labels[c]["tests"][:70] for c in by_claim.index]
by_claim[by_claim[list(VARIANTS)].nunique(axis=1) > 1]

### Read some rationales

A right answer can rest on a wrong reason, so a quick look at a few of them is worth it.

In [ ]:
pd.set_option("display.max_colwidth", 300)
deepseek[(deepseek.variant == BEST) & (deepseek.run == 1)][["claim_id", "decision", "payout", "right", "rationale"]].head(8)

In [ ]:
# Which local answers were slow, and how long they were
local = results[results["where"] == "This computer"]
local = local.reindex(columns=COLUMNS)
local.sort_values("seconds", ascending=False)[["model", "variant", "claim_id", "seconds", "output_tokens", "done_reason", "error"]].head(10)